In [3]:
import pandas as pd
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt

print("--- Step 1: Loading Mine Topology into NetworkX ---")
data_dir = "intern_05_gnn_cycle_time/data/"

# Load exact CSVs per README
df_nodes = pd.read_csv(f"{data_dir}graph_nodes.csv")
df_edges = pd.read_csv(f"{data_dir}graph_edges.csv")
df_trips = pd.read_csv(f"{data_dir}cycle_times.csv")

# Initialize Directed Graph
G = nx.DiGraph()

# Add Nodes with attributes
for _, row in df_nodes.iterrows():
    G.add_node(
        row['node_id'], 
        node_type=row['node_type'],
        pos=(row['x'], row['y']),
        elevation=row['elevation_m'],
        capacity=row['capacity_tph']
    )

# Add Directed Edges with attributes
for _, row in df_edges.iterrows():
    G.add_edge(
        row['from_node'], 
        row['to_node'], 
        edge_id=row['edge_id'],
        distance_km=row['distance_km'],
        gradient_pct=row['gradient_pct'],
        speed_limit=row['speed_limit_kmh']
    )

# Verify Connectivity
is_strongly_connected = nx.is_strongly_connected(G)
is_weakly_connected = nx.is_weakly_connected(G)

print(f"Graph Loaded: {G.number_of_nodes()} Nodes | {G.number_of_edges()} Directed Edges")
print(f"Is Weakly Connected (all nodes reachable without direction)?   {is_weakly_connected}")
print(f"Is Strongly Connected (every node can reach every other node)? {is_strongly_connected}")

# Identify any dead ends or isolated components
if not is_weakly_connected:
    print("WARNING: Graph has disconnected components:", list(nx.weakly_connected_components(G)))
else:
    print("SUCCESS: Mine topology is fully connected and verified for message passing.")

--- Step 1: Loading Mine Topology into NetworkX ---
Graph Loaded: 10 Nodes | 24 Directed Edges
Is Weakly Connected (all nodes reachable without direction)?   True
Is Strongly Connected (every node can reach every other node)? True
SUCCESS: Mine topology is fully connected and verified for message passing.


In [4]:
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, root_mean_squared_error

print("\n--- Step 2: Training Tabular XGBoost Baseline ---")

# 1. Define Tabular Feature Set based on README columns
# We use the route-level aggregates, vehicle state, and temporal/congestion indicators
feature_cols = [
    'loaded', 
    'payload_t', 
    'total_distance_km', 
    'max_gradient_pct', 
    'surface_factor', 
    'shift_hour', 
    'congestion_level', 
    'failure_active'
]

target_col = 'travel_time_min'

# 2. Extract X and y, converting target from minutes to SECONDS for KPI compliance
X = df_trips[feature_cols]
y_seconds = df_trips[target_col] * 60.0

# Strict 80/20 train-test split (using seed 42 for reproducible research)
X_train, X_test, y_train, y_test = train_test_split(
    X, y_seconds, test_size=0.20, random_state=42
)

# 3. Initialize and Train XGBoost Regressor
xgb_baseline = xgb.XGBRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)
xgb_baseline.fit(X_train, y_train)

# 4. Generate Predictions (in seconds)
preds_combined_sec = xgb_baseline.predict(X_test)

# 5. Calculate Combined KPIs
mae_combined = mean_absolute_error(y_test, preds_combined_sec)
rmse_combined = root_mean_squared_error(y_test, preds_combined_sec)

# 6. Calculate Loaded-Only KPIs (where loaded == 1)
loaded_mask = X_test['loaded'] == 1
mae_loaded = mean_absolute_error(y_test[loaded_mask], preds_combined_sec[loaded_mask])
rmse_loaded = root_mean_squared_error(y_test[loaded_mask], preds_combined_sec[loaded_mask])

# 7. Print Official Presentation Table
print("\n" + "="*65)
print("        WEEK 1 DELIVERABLE: TABULAR BASELINE BENCHMARKS        ")
print("="*65)
print(f"Metric Category        | XGBoost Result | Syllabus Target (GNN Goal)")
print("-" * 65)
print(f"Loaded Trips MAE       | {mae_loaded:10.2f} s | ~101 s (GNN Target: <80s)")
print(f"Loaded Trips RMSE      | {rmse_loaded:10.2f} s | Benchmark Reference")
print(f"Combined Trips MAE     | {mae_combined:10.2f} s | ~119 s (GNN Target: <110s)")
print(f"Combined Trips RMSE    | {rmse_combined:10.2f} s | Benchmark Reference")
print("="*65)

# 8. Feature Importance Inspection
importances = pd.Series(xgb_baseline.feature_importances_, index=feature_cols).sort_values(ascending=False)
print("\nTop 5 Most Important Tabular Features (Notice what is missing!):")
print(importances.head(5).to_string())


--- Step 2: Training Tabular XGBoost Baseline ---

        WEEK 1 DELIVERABLE: TABULAR BASELINE BENCHMARKS        
Metric Category        | XGBoost Result | Syllabus Target (GNN Goal)
-----------------------------------------------------------------
Loaded Trips MAE       |      85.76 s | ~101 s (GNN Target: <80s)
Loaded Trips RMSE      |     110.88 s | Benchmark Reference
Combined Trips MAE     |      78.72 s | ~119 s (GNN Target: <110s)
Combined Trips RMSE    |     101.72 s | Benchmark Reference

Top 5 Most Important Tabular Features (Notice what is missing!):
loaded              0.501907
failure_active      0.251595
max_gradient_pct    0.156976
congestion_level    0.033614
payload_t           0.021504
